# Analytical Processing (Gold Layer)

## Create SQL views

In [0]:
%sql
-- Daily prices views
CREATE OR REPLACE VIEW gsma_workspace.global_stocks_schema.gold_daily_prices AS
SELECT *
FROM gsma_workspace.global_stocks_schema.silver_global_stock_data;

-- YTD return views
CREATE OR REPLACE VIEW gsma_workspace.global_stocks_schema.gold_ytd_return AS
WITH ranked_data AS (
    SELECT
        Index,
        Date,
        Close,
        Year,
        FIRST_VALUE(Close) OVER (
            PARTITION BY Index, YEAR(Date)
            ORDER BY Date
        ) AS first_price
    FROM gsma_workspace.global_stocks_schema.silver_global_stock_data
)

SELECT
    Index,
    Year,
    ROUND(MAX(((Close - first_price) / first_price) * 100), 2) AS ytd_return
FROM ranked_data
GROUP BY Index, Year
ORDER BY Index, Year;

-- Average daily return views
CREATE OR REPLACE VIEW gsma_workspace.global_stocks_schema.gold_avg_daily_return AS
SELECT
    Index,
    ROUND(AVG(Daily_Return), 6) AS avg_daily_return
FROM gsma_workspace.global_stocks_schema.silver_global_stock_data
GROUP BY Index;

-- Volatility views
CREATE OR REPLACE VIEW gsma_workspace.global_stocks_schema.gold_volatility AS
SELECT
    Index,
    ROUND(STDDEV(daily_return), 6) AS volatility
FROM gsma_workspace.global_stocks_schema.silver_global_stock_data
GROUP BY Index;

Build multi-year performance views

In [0]:
%sql
CREATE OR REPLACE VIEW gsma_workspace.global_stocks_schema.gold_multi_year_performance AS
SELECT
    Index,
    Year,
    ROUND(AVG(Daily_Return), 6) AS avg_daily_return,
    ROUND(STDDEV(daily_return), 6) AS volatility
FROM gsma_workspace.global_stocks_schema.silver_global_stock_data
GROUP BY Index, Year
ORDER BY Index, Year;

In [0]:
%sql
-- View the results created from views
SELECT * FROM gsma_workspace.global_stocks_schema.gold_daily_prices LIMIT 10;

In [0]:
%sql
SELECT * FROM gsma_workspace.global_stocks_schema.gold_ytd_return LIMIT 10;

In [0]:
%sql
SELECT * FROM gsma_workspace.global_stocks_schema.gold_avg_daily_return LIMIT 10;

In [0]:
%sql
SELECT * FROM gsma_workspace.global_stocks_schema.gold_volatility LIMIT 10;

In [0]:
%sql
SELECT * FROM gsma_workspace.global_stocks_schema.gold_multi_year_performance LIMIT 10;